In [1]:
import torch
from torch import nn
import polars as pl
import numpy as np
from sklearn.preprocessing import StandardScaler
from unicodedata import bidirectional

In [2]:
# ==========================================
# 0 Hyperparameters
# ==========================================

MAX_RUL = 130
WINDOW_SEQ = 32
BATCH_SIZE = 128

#Model
HIDDEN_SIZE = 128
NUM_LAYERS = 1
CNN_FILTERS = 64
CNN_KERNEL = 3
MLP_DROPOUT = 0.4
LSTM_DROPOUT = 0.2

NUM_OF_WORKERS = 0
LR = 1e-3
EPOCHS = 40
WEIGHT_DECAY = 5e-4
T0            = 30         # cosine annealing period
T_MULT        = 2          # cosine annealing multiplier
NOISE_STD = 1e-3
DEVICE = torch.device("cuda")

In [3]:
# ==========================================
# 1 LOAD DATA
# ==========================================
def _validate_join(test_df, test_labels_df):
    n_units_data = test_df.select(["file_path", "unit"]).unique().height
    n_units_labels = test_labels_df.height
    if n_units_data != n_units_labels:
        raise ValueError(
            f"Unit count mismatch after join: {n_units_data} units in test data "
            f"vs {n_units_labels} rows in RUL labels. Check file globbing / ordering."
        )
    if test_df["RUL"].null_count() > 0:
        missing = (
            test_df.filter(pl.col("RUL").is_null())
            .select(["file_path", "unit"])
            .unique()
        )
        raise ValueError(f"RUL join produced nulls for units:\n{missing}")

def _dataset_id_expr(col: str = "file_path") -> pl.Expr:
    """Extract 'FD001' / 'FD002' / etc. from a path regardless of the
    'train_' / 'test_' / 'RUL_' prefix, so different scans can be joined."""
    return pl.col(col).str.extract(r"(FD00\d)", 1).alias("dataset_id")

col_names = ["unit", "cycle"] + [f'op_{i}' for i in range(3)] + [f"s_{i}" for i in range(21)]
drop_cols = ["op_0", "op_1", "op_2", "s_0", "s_4","s_5", "s_9", "s_15", "s_17", "s_18"]
feature_cols = [c for c in col_names if c not in ["unit", "cycle"] + drop_cols]

def load_all_data():
    train_df = (
        pl.scan_csv(
            "CMAPSSData/train_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            new_columns=col_names,
            include_file_paths="file_path",
        )
        .select(col_names + ["file_path"])
        .with_columns(_dataset_id_expr())
        .with_columns(
            (pl.col("cycle").max().over(["dataset_id", "unit"]) - pl.col("cycle"))
            .clip(upper_bound=MAX_RUL)
            .alias("RUL")
        )
        .collect()
    )

    test_labels_df = (
        pl.scan_csv(
            "CMAPSSData/RUL_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            include_file_paths="file_path",
        )
        .with_columns(_dataset_id_expr())
        .sort(["dataset_id"], maintain_order=True)
        .with_columns(pl.int_range(1, pl.len() + 1).over("dataset_id").alias("unit"))
        .select("dataset_id", "unit", pl.col("column_1").alias("true_end_rul"))
        .collect()
    )

    test_df = (
        pl.scan_csv(
            "CMAPSSData/test_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            new_columns=col_names,
            include_file_paths="file_path",
        )
        .select(col_names + ["file_path"])
        .with_columns(_dataset_id_expr())
        .collect()
    )

    return train_df, test_df, test_labels_df

In [4]:
# ==========================================
# 2 Creat windows
# ==========================================

def attach_test_labels(test_df: pl.DataFrame, test_labels_df: pl.DataFrame) -> pl.DataFrame:
    """Return one row per test unit: its last recorded cycle + true_end_rul,
    joined by (dataset_id, unit) — never by position or raw file_path."""

    last_rows = (
        test_df.sort("cycle")
        .group_by(["dataset_id", "unit"], maintain_order=True)
        .last()  # last cycle per unit; keeps all original columns for that row
    )

    last_rows = last_rows.join(
        test_labels_df,
        on=["dataset_id", "unit"],
        how="left",
    )

    _validate_label_join(last_rows)
    return last_rows


def _validate_label_join(last_rows: pl.DataFrame):
    n_null = last_rows["true_end_rul"].null_count()
    if n_null > 0:
        missing = last_rows.filter(pl.col("true_end_rul").is_null()).select(
            "dataset_id", "unit"
        )
        raise ValueError(f"{n_null} test units have no matching RUL label:\n{missing}")

def train_windows(df, feature_cols, window=WINDOW_SEQ):

    X, y = [], []

    for _, group_df in df.group_by(["file_path", "unit"]):
        group_df = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        labels = group_df["RUL"].to_numpy()
        # print(CMAPSSData.shape, labels.shape)
        for i in range(len(group_df)-window+1):
            X.append(data[i:i+window])
            y.append(labels[i+window-1])
        # print(X.shape, y.shape)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def test_windows(test_df: pl.DataFrame, test_labels_df: pl.DataFrame, feature_cols, window=WINDOW_SEQ):
    labels_by_unit = attach_test_labels(test_df, test_labels_df)
    label_lookup = dict(
        zip(
            zip(labels_by_unit["dataset_id"], labels_by_unit["unit"]),
            labels_by_unit["true_end_rul"],
        )
    )

    X, y = [], []
    for (dataset_id, unit), group_df in test_df.group_by(["dataset_id", "unit"], maintain_order=True):
        group_df = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        if len(data) >= window:
            X.append(data[-window:])
        else:
            pad = np.zeros((window - len(data), len(feature_cols)))
            X.append(np.vstack([pad, data]))
        y.append(label_lookup[(dataset_id, unit)])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [5]:
# ==========================================
# 3 Create Datasets
# ==========================================

from torch.utils.data import Dataset, DataLoader

class Train_Dataset(Dataset):
    def __init__(self, X, y, augmentation=True):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        self.augmentation = augmentation

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        x = self.X[index]
        if self.augmentation:
            x = x + torch.randn_like(x) * NOISE_STD
        return x, self.y[index]

class Test_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


In [6]:
# ==========================================
# 4 Model
# ==========================================
class LSTM(nn.Module):
    def __init__(self,
                 n_features,
                 hidden_size = HIDDEN_SIZE,
                 cnn_filters = CNN_FILTERS,
                 cnn_kernel = CNN_KERNEL,
                 mlp_dropout = MLP_DROPOUT,
                 lstm_dropout =LSTM_DROPOUT,
                 num_layers = NUM_LAYERS):
        super().__init__()

        # ======== 1D - CONV ==================
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=n_features, out_channels=cnn_filters, kernel_size=cnn_kernel, padding=cnn_kernel//2),
            nn.BatchNorm1d(cnn_filters),
            nn.GELU(),
            nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters*2, kernel_size=cnn_kernel, padding=cnn_kernel//2),
            nn.BatchNorm1d(cnn_filters*2),
            nn.GELU(),
        )


        # ======== LSTM ==================
        self.lstm = nn.LSTM(
            input_size=cnn_filters*2,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout = lstm_dropout if num_layers > 1 else 0.0
        )

        lstm_out = hidden_size * 2

        # ======== regression - MLP ==================

        self.regressor = nn.Sequential(
            nn.Linear(lstm_out, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(mlp_dropout),
            nn.Linear(128, 1)

        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)

    def forward(self, x):
        #1D-CNN
        h = self.cnn(x.transpose(1, 2))   # (B, C, T)
        h = h.transpose(1, 2)             # (B, T, C)

        # BiLSTM
        lstm_out, (hn, _) = self.lstm(h)  # (B, T, 2H), hn: (2*L, B, H)
        # # Concatenate final forward + backward hidden states
        # last_h = torch.cat([hn[-2], hn[-1]], dim=-1)  # (B, 2H)
        #
        # # Multi-head self-attention (Pre-LN)
        # residual  = lstm_out
        # normed    = self.attn_norm(lstm_out)
        # attn_out, _ = self.mha(normed, normed, normed)
        # lstm_out  = residual + attn_out
        #
        # # Position-wise feed-forward
        # residual  = lstm_out
        # normed    = self.ff_norm(lstm_out)
        # lstm_out  = residual + self.ff(normed)
        #
        # # Global average pool over time (attention-refined)
        # mean_pool = lstm_out.mean(dim=1)   # (B, 2H)
        #
        # # Concatenate and regress
        # out = torch.cat([last_h, mean_pool], dim=-1)  # (B, 4H)
        return self.regressor(lstm_out[:,-1,:]).squeeze(-1)         # (B,)

In [7]:
class gradient_explosion:
    def __init__(self, model):
        self.grad = []
        self.model = model

    def get_grad(self):
        total_norm = 0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        self.grad.append(total_norm ** 0.5)

    def print_grad_norm(self):
        print(f"mean Gradient norm: {np.mean(self.grad):.6f}, max: {np.max(self.grad):.6f}, min: {np.min(self.grad):.6f}")


In [8]:
# ==========================================
# 5 Trainer function
# ==========================================
def MSE(y_hat, y):
    return torch.mean((y_hat - y) ** 2)


class Trainer:
    def __init__(self, model, train_dataloader, test_dataloader, optimizer, scheduler, device, loss_fn, epoch, check_gradient = False):
        self.model = model
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.loss_fn = loss_fn          # e.g., MSE for training
        self.epoch = epoch
        if check_gradient:
            self.gradient_checker = gradient_explosion(self.model)
        else:
            self.gradient_checker = None

    def fit_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_samples = 0
        total_squared_error = 0.0

        for X, y in self.train_dataloader:
            X, y = X.to(self.device), y.to(self.device)

            self.optimizer.zero_grad(set_to_none=True)
            pred = self.model(X).squeeze()   # ensure shape (batch,)
            loss = self.loss_fn(pred, y)     # MSE
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm =  1.0)
            self.optimizer.step()
            self.gradient_checker.get_grad()
            batch_size = y.size(0)
            total_loss += loss.item() * batch_size          # sum of MSE losses (weighted by batch size)
            total_squared_error += torch.sum((pred - y) ** 2).item()
            total_samples += batch_size

        # Return average MSE and RMSE for the epoch (optional)
        avg_mse = total_loss / total_samples
        rmse = np.sqrt(total_squared_error / total_samples)
        self.gradient_checker.print_grad_norm()
        return avg_mse, rmse

    def validate_epoch(self):
        self.model.eval()
        total_squared_error = 0.0
        total_samples = 0

        with torch.no_grad():
            for X, y in self.test_dataloader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X).squeeze()
                total_squared_error += torch.sum((pred - y) ** 2).item()
                total_samples += y.size(0)

        rmse = np.sqrt(total_squared_error / total_samples)
        return rmse

    def fit(self):
        for epoch in range(self.epoch):
            train_mse, train_rmse = self.fit_epoch()
            val_rmse = self.validate_epoch()
            self.scheduler.step()

            print(f"Epoch {epoch+1}/{self.epoch} | Train MSE: {train_mse:.4f} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

In [9]:
# Load
if __name__ == "__main__":

    train_df, test_df, RUL_test = load_all_data()
    train_X, train_y = train_windows(train_df, feature_cols)
    test_X, test_y = test_windows(test_df, RUL_test, feature_cols)

    scaler = StandardScaler().fit(train_X.reshape(-1, len(feature_cols)))
    train_X = scaler.transform(train_X.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))
    test_X  = scaler.transform(test_X.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))

    train_dataloader = DataLoader(Train_Dataset(train_X, train_y), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_OF_WORKERS, pin_memory=True, drop_last=True)
    test_dataloader = DataLoader(Test_Dataset(test_X, test_y), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_OF_WORKERS, pin_memory=True)

    loss_fn = MSE
    # model = LSTM(n_features=len(feature_cols), hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, cnn_filters=CNN_FILTERS, cnn_kernel=CNN_KERNEL, lstm_bidirectional=True, lstm_dropout = 0.2, mlp_dropout=0.4)
    model = LSTM(len(feature_cols))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    # Cosine Annealing with Warm Restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimiser, T_0=T0, T_mult=T_MULT, eta_min=LR * 0.01
    )

    trainer = Trainer(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        optimizer=optimiser,
        scheduler=scheduler,
        device=device,
        loss_fn=loss_fn,
        epoch=EPOCHS,
        check_gradient=True
    )

    trainer.fit()

mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 1/40 | Train MSE: 2571.3866 | Train RMSE: 50.7088 | Val RMSE: 32.8367
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 2/40 | Train MSE: 596.4334 | Train RMSE: 24.4220 | Val RMSE: 28.4169
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 3/40 | Train MSE: 467.3377 | Train RMSE: 21.6180 | Val RMSE: 28.7843
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 4/40 | Train MSE: 409.9907 | Train RMSE: 20.2482 | Val RMSE: 27.3518
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 5/40 | Train MSE: 358.6548 | Train RMSE: 18.9382 | Val RMSE: 26.8821
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 6/40 | Train MSE: 300.3546 | Train RMSE: 17.3307 | Val RMSE: 26.0141
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Epoch 7/40 | Train MSE: 254.1269 | Train RMSE: 15.9414 | Val RMSE: 26.4583
mean Gradient norm: 1.000000, max: 1.000000, min: 1.000000
Ep